# Фінальний проєкт. FitCoach – фітнес-асистент
**Курс:** AI Fundamentals · Product Management · Катерина Карпова

**Для кого:** жінки 40–45 років із сидячою роботою, яких турбує біль у спині чи шиї і які бояться собі нашкодити.

**Що вміє:** рахує ІМТ і норму калорій, підбирає безпечні вправи для дому й складає план тренування на потрібну кількість хвилин.

**Як влаштовано:** граф LangGraph. Дані профілю й тривожні ознаки розбирає код, модель лише обирає інструмент і формулює відповідь.

**Основа:** ноутбук теми 8 (ДЗ4). Попередня версія збережена в історії репозиторію.

⚡ **Запуск:** ключ `OPENAI_API_KEY` у Colab Secrets (або у файлі `.env` локально), далі комірки зверху вниз.

## 0. Налаштування

In [1]:
# Встановлення бібліотек потрібне лише в Google Colab: локально вони вже є у віртуальному середовищі.
# Версії зафіксовані (ті самі, що в requirements.txt), щоб ноутбук поводився однаково всюди.
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "langchain==1.3.11", "langchain-openai==1.3.3", "langgraph==1.2.7",
         "python-dotenv==1.2.2", "gradio==6.32.0"],
        check=True,
    )
    print("✅ Бібліотеки встановлено в Colab")
else:
    print("✅ Локальний запуск: бібліотеки з віртуального середовища")

✅ Локальний запуск: бібліотеки з віртуального середовища


In [2]:
import os
import getpass

from langchain_openai import ChatOpenAI

# Ключ: у Colab з Secrets, локально з файлу .env; якщо ніде немає, попросимо ввести (ключ не виводиться)
if IN_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv(".env")

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# Та сама модель і параметри, що в ДЗ4: так різницю в поведінці можна приписати архітектурі, а не моделі
MODEL_NAME = "gpt-4o-mini"
chat_model = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0.3,
    max_tokens=512,
    timeout=30,
)

check = chat_model.invoke("Одним коротким реченням підтвердь, що ти готовий працювати.")
print(f"✅ Модель {MODEL_NAME} відповідає: {check.content}")

✅ Модель gpt-4o-mini відповідає: Я готовий працювати!


## 1. Дані

База вправ для дому й рівні активності. Протипоказання розмічені за загальнодоступними рекомендаціями; перед продакшном їх має перевірити фізіотерапевт.

In [4]:
# База вправ для дому. Ключі груп англійською, "label" — назва для людини,
# "aliases" — основи слів, за якими код упізнає групу в будь-якому відмінку ("спин" → "спина", "спину").
# minutes — орієнтовна тривалість вправи разом з відпочинком (для плану тренування).
# contraindications — стани, за яких вправу не пропонуємо:
#   lower_back_pain — біль у попереку, neck_pain — біль у шиї, acute_back_injury — гостра травма спини.
# Розминка й заминка — окремі групи: їх бере план тренування для початку й кінця.

def ex(name, level, technique, sets, reps, minutes, contraindications=()):
    return {"name": name, "level": level, "technique": technique, "sets": sets, "reps": reps,
            "minutes": minutes, "contraindications": list(contraindications)}

T, P, S = "acute_back_injury", "lower_back_pain", "neck_pain"

EXERCISES_DB = {
    "warmup": {"label": "розминка", "aliases": ["розминк", "розігрів"], "exercises": [
        ex("Ходьба на місці", "beginner", "Крокуй на місці в комфортному темпі, руки рухаються вільно", "1", "2 хвилини", 2),
        ex("Кругові рухи плечима", "beginner", "Повільно обертай плечима назад, потім уперед", "1", "10 у кожен бік", 1),
        ex("Махи руками перед грудьми", "beginner", "Розведи руки в сторони й плавно схрещуй їх перед собою", "1", "15", 1),
        ex("Підйоми на носки", "beginner", "Стоячи біля опори, плавно підіймайся на носки й опускайся", "1", "15", 1),
        ex("Кругові рухи тазом", "beginner", "Руки на поясі, повільно описуй тазом невеликі кола", "1", "8 у кожен бік", 1, [T]),
        ex("М'які повороти тулуба стоячи", "beginner", "Стопи на ширині плечей, плавно повертай корпус вправо й уліво без ривків", "1", "10 у кожен бік", 1, [T]),
    ]},
    "back": {"label": "спина", "aliases": ["спин", "попере"], "exercises": [
        ex("Котик-корова", "beginner", "Стоячи на четвереньках, по черзі плавно прогинай і округлюй спину", "2", "10", 2, [T]),
        ex("Нахили таза лежачи", "beginner", "Лежачи на спині, ноги зігнуті, м'яко притискай поперек до підлоги й розслабляй", "2", "10", 2, [T]),
        ex("Сідничний міст", "beginner", "Лежачи на спині, ноги зігнуті, плавно підіймай таз і опускай", "2-3", "10-12", 3, [T]),
        ex("Птах-собака", "beginner", "На четвереньках витягни протилежні руку й ногу, затримайся на 3 секунди, поміняй сторону", "2", "8 на кожну сторону", 3, [T]),
        ex("Сфінкс", "intermediate", "Лежачи на животі, обіприся на передпліччя й м'яко підніми груди, затримайся на 10 секунд", "2", "5", 2, [P, T]),
        ex("Супермен", "intermediate", "Лежачи на животі, одночасно підійми руки й ноги, затримайся на 2-3 секунди", "3", "10", 3, [P, T]),
    ]},
    "chest": {"label": "груди", "aliases": ["груд"], "exercises": [
        ex("Віджимання від стіни", "beginner", "Стань на крок від стіни, долоні на рівні плечей, повільно згинай і розгинай руки", "2-3", "10-12", 3),
        ex("Стискання долонь на рівні грудей", "beginner", "Сидячи рівно, з'єднай долоні на рівні грудей і стискай їх 10 секунд", "2", "5", 1.5),
        ex("Віджимання від столу", "beginner", "Упрись руками в край стійкого столу, тіло рівне, опускай груди до краю", "2-3", "8-10", 3, [T]),
        ex("Віджимання з колін", "beginner", "Коліна на підлозі, тіло рівне від колін до голови, опускай груди до підлоги", "2-3", "8-10", 3, [T]),
        ex("Віджимання від підлоги", "intermediate", "Руки на ширині плечей, тіло пряме, опускайся до торкання грудьми підлоги", "3", "10-15", 3, [T]),
    ]},
    "legs": {"label": "ноги", "aliases": ["ног", "ніг", "стегн", "сідниц"], "exercises": [
        ex("Присідання на стілець", "beginner", "Повільно сідай на стілець і вставай без допомоги рук, спина пряма", "2-3", "10", 3),
        ex("Підйоми на носки біля стіни", "beginner", "Тримаючись за стіну, підіймайся на носки й повільно опускайся", "2", "15", 2),
        ex("Відведення ноги вбік з опорою", "beginner", "Тримаючись за спинку стільця, плавно відводь пряму ногу вбік", "2", "10 на кожну ногу", 2.5),
        ex("Присідання", "beginner", "Стопи на ширині плечей, спина пряма, присідай до паралелі стегон з підлогою", "3", "12-15", 3, [T]),
        ex("Сидіння біля стіни", "intermediate", "Спиною до стіни, сповзи вниз до кута 90 градусів у колінах і тримай 20-30 секунд", "3", "20-30 секунд", 2, [T]),
        ex("Випади назад з опорою", "intermediate", "Тримаючись за опору, зроби крок назад і м'яко опусти коліно до підлоги", "3", "8 на кожну ногу", 3, [T]),
    ]},
    "core": {"label": "прес", "aliases": ["прес", "живіт", "живот"], "exercises": [
        ex("Планка біля стіни", "beginner", "Упрись передпліччями в стіну, тіло рівне під кутом, тримай 20 секунд", "3", "20 секунд", 2, [T]),
        ex("Мертвий жук", "beginner", "Лежачи на спині, руки й ноги вгору, по черзі опускай протилежні руку й ногу, поперек притиснутий", "2", "8 на кожну сторону", 3, [T]),
        ex("Планка з колін", "beginner", "Упор на передпліччя й коліна, тіло рівне, тримай 15-20 секунд", "3", "15-20 секунд", 2, [T]),
        ex("Бічна планка з колін", "beginner", "Лежачи на боці, обіприся на передпліччя й коліно, підніми таз і тримай 15 секунд", "2", "15 секунд на кожен бік", 2, [T]),
        ex("Планка", "intermediate", "Упор на передпліччя й носки, тіло рівне від голови до п'ят, тримай 20-30 секунд", "3", "20-30 секунд", 2, [T]),
        ex("Скручування", "intermediate", "Лежачи на спині, ноги зігнуті, підіймай плечі від підлоги", "3", "15", 3, [P, S, T]),
    ]},
    "neck_shoulders": {"label": "шия і плечі", "aliases": ["шия", "шию", "шиї", "шиєю", "плеч", "лопат"], "exercises": [
        ex("Втягування підборіддя", "beginner", "Сидячи рівно, м'яко відведи підборіддя назад, ніби робиш подвійне підборіддя, затримайся на 3 секунди", "2", "10", 1.5),
        ex("Зведення лопаток сидячи", "beginner", "Сидячи рівно, зведи лопатки разом, затримайся на 3 секунди, розслаб", "2", "10", 1.5),
        ex("Підйоми плечей", "beginner", "Повільно підніми плечі до вух, затримайся на 2 секунди й опусти", "2", "10", 1.5),
        ex("Ангели біля стіни", "beginner", "Спиною до стіни, руки зігнуті й притиснуті, повільно ковзай ними вгору й униз", "2", "10", 2),
        ex("Нахили голови до плеча", "beginner", "Повільно нахили голову до плеча, утримуй 15 секунд, без ривків", "2", "по 15 секунд у кожен бік", 1.5, [T]),
        ex("Жим рук угору з пляшками води", "intermediate", "Сидячи, тримай пляшки на рівні плечей і плавно піднімай руки вгору", "3", "10", 2.5, [S, T]),
    ]},
    "arms": {"label": "руки", "aliases": ["рук", "біцепс", "трицепс"], "exercises": [
        ex("Згинання рук з пляшками води", "beginner", "Стоячи чи сидячи, лікті притиснуті до тулуба, згинай руки з пляшками до плечей", "2-3", "12", 2.5),
        ex("Молоток з пляшками води", "beginner", "Тримай пляшки вертикально, як молоток, і згинай руки до плечей", "2-3", "12", 2.5),
        ex("Підйоми рук уперед з пляшками", "beginner", "Тримаючи пляшки, плавно піднімай прямі руки вперед до рівня плечей", "2", "10", 2),
        ex("Ізометричне згинання з рушником", "beginner", "Стань на середину рушника, тягни кінці вгору зігнутими руками й тримай 10 секунд", "2", "5", 1.5),
        ex("Розгинання рук з-за голови з пляшкою", "intermediate", "Сидячи, тримай пляшку двома руками за головою й розгинай руки вгору", "2-3", "10", 2.5, [S]),
    ]},
    "cooldown": {"label": "заминка", "aliases": ["заминк", "розтяжк", "розтягн"], "exercises": [
        ex("Дихання животом лежачи", "beginner", "Лежачи на спині, долоня на животі, повільно вдихай носом і видихай ротом", "1", "2 хвилини", 2),
        ex("Розтяжка грудних м'язів у дверному отворі", "beginner", "Поклади передпліччя на одвірок і м'яко подайся вперед до легкого натягу, тримай 20 секунд", "1", "20 секунд на кожен бік", 1),
        ex("Розтяжка стегна стоячи з опорою", "beginner", "Тримаючись за опору, зігни ногу й підтягни п'яту до сідниці, тримай 20 секунд", "1", "20 секунд на кожну ногу", 1.5),
        ex("Розтяжка задньої поверхні стегна на стільці", "beginner", "Сидячи на краю стільця, випрями одну ногу й з рівною спиною трохи нахилися вперед", "1", "20 секунд на кожну ногу", 1.5, [T]),
        ex("Поза дитини", "beginner", "Сядь на п'яти, витягни руки вперед і опусти груди до стегон, дихай спокійно", "1", "30 секунд", 1.5, [T]),
        ex("Коліна до грудей лежачи", "beginner", "Лежачи на спині, обійми коліна й м'яко підтягни їх до грудей, тримай 20 секунд", "1", "20 секунд", 1.5, [T]),
    ]},
}

total = sum(len(g["exercises"]) for g in EXERCISES_DB.values())
safe_back = sum(1 for g in EXERCISES_DB.values() for e in g["exercises"] if P not in e["contraindications"])
print(f"✅ База: {len(EXERCISES_DB)} груп, {total} вправ для дому, з них {safe_back} доступні при болю в попереку")

✅ База: 8 груп, 46 вправ для дому, з них 43 доступні при болю в попереку


In [5]:
# Рівні активності для формули калорій (з ДЗ4).
# Ключі англійською, "factor" — коефіцієнт для денної норми, "label" — назва для людини,
# "aliases" — основи слів, за якими код упізнає рівень у відповіді ("помірн" → "помірна", "помірною").
ACTIVITY_LEVELS = {
    "sedentary":   {"factor": 1.2,   "label": "мінімальна",   "aliases": ["мінімальн", "сидяч", "сидж"]},
    "light":       {"factor": 1.375, "label": "легка",        "aliases": ["легк"]},
    "moderate":    {"factor": 1.55,  "label": "помірна",      "aliases": ["помірн", "середн"]},
    "active":      {"factor": 1.725, "label": "висока",       "aliases": ["висок"]},
    "very_active": {"factor": 1.9,   "label": "екстремальна", "aliases": ["екстремальн"]},
}

print(f"✅ Рівні активності: {', '.join(level['label'] for level in ACTIVITY_LEVELS.values())}")

✅ Рівні активності: мінімальна, легка, помірна, висока, екстремальна


## 2. Профіль: дані розбирає код

Модель не передає в інструменти ні біль, ні стать, ні вагу: усе це код записує в профіль зі слів людини (або з форми в інтерфейсі). Якщо даних бракує, код ставить одне питання за раз у фіксованому порядку.

In [6]:
import re

# ============================================================
# Розпізнавання даних у тексті людини (без моделі)
# ============================================================

def all_numbers(text: str) -> list[float]:
    """Усі числа в тексті, разом зі знаком мінус ("-72" не стане 72)."""
    return [float(n.replace(",", ".")) for n in re.findall(r"-?\d+(?:[.,]\d+)?", text)]


def labelled_number(text: str, stems: list[str]) -> float | None:
    """Число одразу після слова з однією з основ: "вага 72", "важу 72", "зріст 165"."""
    for stem in stems:
        match = re.search(stem + r"\w*\D{0,10}?(-?\d+(?:[.,]\d+)?)", text)
        if match:
            return float(match.group(1).replace(",", "."))
    return None


def clauses(text: str) -> list[str]:
    """Ділить повідомлення на частини за розділовими знаками й "але",
    щоб заперечення стосувалося лише своєї частини: "болить поперек, але оніміння немає"."""
    return [c.strip() for c in re.split(r"[.,;!?\n]|\bале\b|\bа\b", text) if c.strip()]


NEGATION = r"\b(не|нема|немає|без|ні|нічого)\b"

# Тривожні ознаки (з ДЗ2): за них вправ не підбираємо, а радимо лікаря.
# "німі" ловить "німіє", "оніміння"; \b не дає "нічн" спрацювати всередині слова "технічно".
RED_FLAG_PATTERNS = [
    r"віддає\s+[ву]\s+(ног|рук|сідниц)",
    r"німі",
    r"\bнічн",
    r"вночі|уночі|ночами",
    r"травм",
    r"\bвпа(в|ла)\b",
    r"падін",
]


def red_flag_status(text: str) -> str | None:
    """'confirmed', якщо в якійсь частині повідомлення є тривожна ознака без заперечення;
    'denied', якщо ознаки згадано лише із запереченням ("оніміння немає", "травм не було");
    None, якщо про ознаки нічого не сказано."""
    mentioned, confirmed = False, False
    for part in clauses(text):
        if any(re.search(p, part) for p in RED_FLAG_PATTERNS):
            mentioned = True
            if not re.search(NEGATION, part):
                confirmed = True
    if confirmed:
        return "confirmed"
    return "denied" if mentioned else None


PAIN_WORDS = ["бол", "біль", "ниє", "ниють", "тягне", "ломить", "скут"]
PAIN_ZONES = {
    "lower_back": ["попере", "спин", "поясниц"],
    "neck": ["шия", "шию", "шиї", "шиєю"],
}


def pain_zones(text: str) -> set[str]:
    """Зони болю, якщо в частині повідомлення є і слово про біль, і зона без заперечення.
    "Вправи на спину" — не біль; "болить поперек" — біль; "спина не болить" — не біль."""
    zones = set()
    for part in clauses(text):
        if any(w in part for w in PAIN_WORDS) and not re.search(NEGATION, part):
            zones |= {zone for zone, words in PAIN_ZONES.items() if any(w in part for w in words)}
    return zones


def parse_sex(text: str) -> str | None:
    # Жіночі слова перевіряємо першими, бо "female" містить "male" (помилка шаблону, знайдена в ДЗ4)
    if re.search(r"жін|дівч|female|^ж$", text):
        return "female"
    if re.search(r"чолов|хлоп|male|^ч$", text):
        return "male"
    return None


def parse_activity(text: str) -> str | None:
    return next((key for key, level in ACTIVITY_LEVELS.items()
                 if any(alias in text for alias in level["aliases"])), None)


def parse_goal(text: str) -> str | None:
    if re.search(r"швидк", text) and re.search(r"схуд|скинут|зменш", text):
        return "fast_loss"
    if re.search(r"схуд|скинут|зменш", text):
        return "gentle_loss"
    if re.search(r"набр|набира|збільш|м'яз|мяз", text):
        return "gain"
    if re.search(r"підтрим|зберег|утрим", text):
        return "maintain"
    return None


def parse_yes_no(text: str) -> bool | None:
    if re.search(r"^\s*(ні|не)\b|не хочу|не треба|не згод", text):
        return False
    if re.search(r"^\s*(так|ок|добре|згодна|згоден|можна|звісно)\b|згод", text):
        return True
    return None


print("✅ Функції розпізнавання готові")

✅ Функції розпізнавання готові


In [7]:
# ============================================================
# Профіль, питання і порядок
# ============================================================

def empty_profile() -> dict:
    return {
        "height_cm": None, "weight_kg": None, "age": None,
        "sex": None, "activity": None, "goal": None,
        "pain_areas": [],              # "lower_back", "neck"
        "health_consent": None,        # згода зберігати дані про біль: True / False / None (ще не питали)
        "red_flags": "not_checked",    # "not_checked" / "denied" / "confirmed"
    }

# У такому порядку код питає дані, яких бракує: зріст перед вагою, бо вага для багатьох чутлива тема
QUESTION_ORDER = ["height_cm", "weight_kg", "age", "sex", "activity", "goal", "health_consent", "red_flags"]

QUESTIONS = {
    "height_cm": "Який у тебе зріст у сантиметрах? Наприклад, 165.",
    "weight_kg": "А вага в кілограмах? Вона потрібна лише для розрахунку, без жодних оцінок.",
    "age": "Скільки тобі років?",
    "sex": "Для формули калорій потрібна біологічна стать: жінка чи чоловік?",
    "activity": ("Яка в тебе щоденна активність? Мінімальна (сидяча робота), легка, "
                 "помірна, висока чи екстремальна?"),
    "goal": "Яка твоя мета: схуднути, підтримувати вагу чи набрати?",
    "health_consent": ("Ти згадуєш біль. Можна зберегти це в профілі, щоб надалі пропонувати "
                       "лише безпечні вправи? Так чи ні."),
    "red_flags": ("Щоб підібрати вправи, які не нашкодять, уточню: чи віддає біль у ногу чи руку, "
                  "чи є оніміння, чи болить уночі, чи була травма або падіння?"),
}

# Допустимі діапазони: число поза ними не записуємо, а перепитуємо
RANGES = {"height_cm": (100, 250), "weight_kg": (30, 300), "age": (10, 110)}
UNITS = {"height_cm": "см", "weight_kg": "кг", "age": "роках"}
# Підписані числа: число йде ПІСЛЯ слова ("вага 72", "зріст 165", "вік 43")
LABELS = {"height_cm": ["зріст", "зрост"], "weight_kg": ["ваг", "важу"], "age": ["вік"]}
NAMES_WHAT = {"height_cm": "зріст", "weight_kg": "вага", "age": "вік"}    # "це вага"
NAMES_ASK = {"height_cm": "зріст", "weight_kg": "вагу", "age": "вік"}     # "напиши вагу"


def reask_number(field: str, value: float) -> str:
    """М'яке перепитування, якщо число поза діапазоном. Підказує, якщо схоже на зріст чи вагу."""
    for other in ("height_cm", "weight_kg"):
        low, high = RANGES[other]
        if other != field and low <= value <= high:
            return (f"Схоже, {value:g} — це {NAMES_WHAT[other]}. "
                    f"Напиши, будь ласка, {NAMES_ASK[field]} у {UNITS[field]}.")
    return (f"Число {value:g} виглядає незвично для цього питання. "
            f"Перевір, будь ласка, і напиши {NAMES_ASK[field]} ще раз.")


def update_profile(profile: dict, awaiting: str | None, message: str) -> tuple[dict, str | None, str | None]:
    """Оновлює профіль за повідомленням людини.
    Повертає (профіль, нове awaiting, текст перепитування або None).
    Тривожні ознаки, біль і підписані числа шукає в кожному повідомленні,
    а непідписану відповідь читає як відповідь на те питання, якого чекає (awaiting)."""
    profile = dict(profile)
    text = message.lower().strip()

    # 1. Тривожні ознаки — у кожному повідомленні
    status = red_flag_status(text)
    if status == "confirmed":
        profile["red_flags"] = "confirmed"
    elif status == "denied" and profile["red_flags"] != "confirmed":
        profile["red_flags"] = "denied"

    # 2. Біль: зони записуємо одразу, бо від них залежить безпека вправ у цій розмові
    new_zones = pain_zones(text) - set(profile["pain_areas"])
    if new_zones:
        profile["pain_areas"] = sorted(set(profile["pain_areas"]) | new_zones)

    # 3. Підписані числа в будь-якому повідомленні
    labelled = set()
    for field, stems in LABELS.items():
        value = labelled_number(text, stems)
        if value is not None and RANGES[field][0] <= value <= RANGES[field][1]:
            profile[field] = value
            labelled.add(field)

    # 4. Відповідь на питання, якого чекаємо
    if awaiting in RANGES:
        if profile[awaiting] is None:
            # Число, підписане як інше поле ("зріст 165", коли чекаємо вагу), не є відповіддю на це питання
            if labelled:
                return profile, awaiting, f"Дякую. А тепер напиши, будь ласка, {NAMES_ASK[awaiting]}."
            numbers = all_numbers(text)
            if len(numbers) == 1:
                value = numbers[0]
                low, high = RANGES[awaiting]
                if low <= value <= high:
                    profile[awaiting] = value
                else:
                    return profile, awaiting, reask_number(awaiting, value)
            elif len(text.split()) <= 3:
                return profile, awaiting, f"Напиши, будь ласка, {NAMES_ASK[awaiting]} одним числом."
        awaiting = None

    elif awaiting in ("sex", "activity", "goal"):
        parser = {"sex": parse_sex, "activity": parse_activity, "goal": parse_goal}[awaiting]
        value = parser(text)
        if value:
            profile[awaiting] = value
            awaiting = None
        elif len(text.split()) <= 3:
            return profile, awaiting, QUESTIONS[awaiting]
        else:
            awaiting = None   # схоже на нове питання, а не на відповідь: не перепитуємо

    elif awaiting == "health_consent":
        answer = parse_yes_no(text)
        if answer is not None:
            profile["health_consent"] = answer
            awaiting = None
        else:
            return profile, awaiting, "Відповідай, будь ласка, «так» чи «ні»."

    elif awaiting == "red_flags":
        if profile["red_flags"] == "not_checked":
            answer = parse_yes_no(text)
            if answer is False or re.search(r"нічого|немає|нема", text):
                profile["red_flags"] = "denied"
            elif answer is True:
                profile["red_flags"] = "confirmed"   # "так" на питання про ознаки: обережно вважаємо, що є
            else:
                return profile, awaiting, QUESTIONS["red_flags"]
        awaiting = None

    return profile, awaiting, None


print("✅ Профіль і питання готові")

✅ Профіль і питання готові


In [8]:
# ============================================================
# Пряма перевірка розбору профілю без моделі
# Кожен кейс: (опис, на яке питання чекаємо, повідомлення, очікування)
# ============================================================
PROFILE_CHECKS = [
    ("Відповідь на питання про зріст", "height_cm", "165",
     {"height_cm": 165.0, "awaiting": None}),
    ("Вага замість зросту: м'яко перепитати", "height_cm", "72",
     {"height_cm": None, "reply_contains": "це вага"}),
    ("Баг з ДЗ5: «зріст 165», коли чекаємо вагу", "weight_kg", "зріст 165",
     {"height_cm": 165.0, "weight_kg": None, "reply_contains": "вагу"}),
    ("Мінус у вазі не стає 72", "weight_kg", "-72",
     {"weight_kg": None, "reply_contains": "незвично"}),
    ("Підписані числа без питання", None, "Мені 43, вага 72, зріст 165",
     {"weight_kg": 72.0, "height_cm": 165.0}),
    ("Біль і заперечення ознаки в одному реченні", None, "Болить поперек, але оніміння немає",
     {"pain_areas": ["lower_back"], "red_flags": "denied"}),
    ("Тривожна ознака посеред розмови", None, "Тепер ще почала німіти нога",
     {"red_flags": "confirmed"}),
    ("Падіння окремо від заперечення травми", None, "Впала вчора, без травм",
     {"red_flags": "confirmed"}),
    ("«технічно» не є ознакою «нічний біль»", None, "Технічно все добре",
     {"red_flags": "not_checked"}),
    ("Запит про вправи на спину — не біль", None, "Які вправи на спину вдома?",
     {"pain_areas": []}),
    ("Відповідь «просто ниє» на питання про ознаки", "red_flags", "Ні, нічого такого, просто ниє після роботи",
     {"red_flags": "denied"}),
    ("Стать: «female» не стає чоловіком", "sex", "female",
     {"sex": "female"}),
    ("Активність своїми словами", "activity", "сиджу весь день",
     {"activity": "sedentary"}),
    ("Мета своїми словами", "goal", "хочу схуднути",
     {"goal": "gentle_loss"}),
    ("Згода на збереження даних про біль", "health_consent", "так",
     {"health_consent": True}),
]

passed = 0
for title, awaiting, message, expected in PROFILE_CHECKS:
    profile, new_awaiting, reply = update_profile(empty_profile(), awaiting, message)
    problems = []
    for key, value in expected.items():
        if key == "awaiting":
            if new_awaiting != value:
                problems.append(f"awaiting={new_awaiting}")
        elif key == "reply_contains":
            if not reply or value not in reply:
                problems.append(f"відповідь: {reply!r}")
        elif profile[key] != value:
            problems.append(f"{key}={profile[key]!r}")
    ok = not problems
    passed += ok
    print(f"{'✅' if ok else '❌'} {title}: «{message}»" + ("" if ok else f" → {'; '.join(problems)}"))
    if reply:
        print(f"     💬 {reply}")

print(f"\nПройдено {passed} з {len(PROFILE_CHECKS)}")

✅ Відповідь на питання про зріст: «165»
✅ Вага замість зросту: м'яко перепитати: «72»
     💬 Схоже, 72 — це вага. Напиши, будь ласка, зріст у см.
✅ Баг з ДЗ5: «зріст 165», коли чекаємо вагу: «зріст 165»
     💬 Дякую. А тепер напиши, будь ласка, вагу.
✅ Мінус у вазі не стає 72: «-72»
     💬 Число -72 виглядає незвично для цього питання. Перевір, будь ласка, і напиши вагу ще раз.
✅ Підписані числа без питання: «Мені 43, вага 72, зріст 165»
✅ Біль і заперечення ознаки в одному реченні: «Болить поперек, але оніміння немає»
✅ Тривожна ознака посеред розмови: «Тепер ще почала німіти нога»
✅ Падіння окремо від заперечення травми: «Впала вчора, без травм»
✅ «технічно» не є ознакою «нічний біль»: «Технічно все добре»
✅ Запит про вправи на спину — не біль: «Які вправи на спину вдома?»
✅ Відповідь «просто ниє» на питання про ознаки: «Ні, нічого такого, просто ниє після роботи»
✅ Стать: «female» не стає чоловіком: «female»
✅ Активність своїми словами: «сиджу весь день»
✅ Мета своїми словами: «хочу